<a href="https://colab.research.google.com/github/ivandalss/amazon-ads-data-pipeline/blob/main/notebooks/meridian_mmm_amazon_ads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install google-meridian
import meridian
print(meridian.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 51.9 MB/s eta 0:00:00
  Attempting uninstall: natsort
    Found existing installation: natsort 8.4.0
    Uninstalling natsort-8.4.0:
      Successfully uninstalled natsort-8.4.0
  Attempting uninstall: arviz
    Found existing installation: arviz 0.22.0
    Uninstalling arviz-0.22.0:
      Successfully uninstalled arviz-0.22.0
1.7.0


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import meridian
from meridian import data as meridian_data
from meridian.model import model, spec
from meridian.model import prior_distribution
import tensorflow_probability as tfp

print("Meridian version:", meridian.__version__)
print("All imports OK")


Meridian version: 1.7.0
All imports OK


In [6]:
# Crear dataset sintetico para Meridian
np.random.seed(42)
n_weeks = 104

# Spend por canal (weekly, en USD)
sp_spend  = np.random.gamma(3, 1000, n_weeks)
sb_spend  = np.random.gamma(2, 500,  n_weeks)
dsp_spend = np.random.gamma(1.5, 800, n_weeks)

# Estacionalidad (Q4 boost)
weeks = np.arange(n_weeks)
seasonality = 1 + 0.3 * np.sin(2 * np.pi * weeks / 52 - np.pi/2)

# Ventas con efectos conocidos (ground truth)
baseline = 50000
sales = (
    baseline * seasonality
    + 3.5 * sp_spend
    + 2.0 * sb_spend
    + 1.2 * dsp_spend
    + np.random.normal(0, 2000, n_weeks)
)

# Armar DataFrame
df = pd.DataFrame({
    'week':        pd.date_range('2023-01-02', periods=n_weeks, freq='W'),
    'sales':       sales,
    'sp_spend':    sp_spend,
    'sb_spend':    sb_spend,
    'dsp_spend':   dsp_spend,
    'seasonality': seasonality
})

print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nSales range: ${df['sales'].min():,.0f} -- ${df['sales'].max():,.0f}")
print(f"Total SP spend: ${df['sp_spend'].sum():,.0f}")

        week         sales     sp_spend     sb_spend    dsp_spend  seasonality
0 2023-01-08  52152.460987  3562.818663   784.548598   917.427378     0.700000
1 2023-01-15  45514.345873  2447.194398  1073.512973   705.463653     0.702187
2 2023-01-22  48337.813756  2302.280567   841.754225  1240.888633     0.708717
3 2023-01-29  46571.511027  2302.304875  2167.763608   385.478413     0.719495
4 2023-02-05  62982.138033  6166.139937  1305.956418   842.472872     0.734363

Shape: (104, 6)

Sales range: $42,728 -- $91,011
Total SP spend: $304,139


In [7]:
# Configurar input data para Meridian
from meridian.data import test_utils

# Meridian necesita los datos en formato de arrays con dimensiones especificas
# n_times = semanas, n_geos = 1 (nacional), n_media_channels = 3

n_times = len(df)
n_geos = 1

# KPI (ventas) - shape: (n_geos, n_times)
kpi = df['sales'].values.reshape(1, n_times)

# Media spend - shape: (n_geos, n_times, n_channels)
media_spend = np.stack([
    df['sp_spend'].values,
    df['sb_spend'].values,
    df['dsp_spend'].values
], axis=-1).reshape(1, n_times, 3)

# Media data (usamos spend como proxy de impressions)
media = media_spend.copy()

# Nombres de canales
channel_names = ['Sponsored_Products', 'Sponsored_Brands', 'DSP']

print("KPI shape:", kpi.shape)
print("Media shape:", media.shape)
print("Channels:", channel_names)
print("\nAll shapes OK")


KPI shape: (1, 104)
Media shape: (1, 104, 3)
Channels: ['Sponsored_Products', 'Sponsored_Brands', 'DSP']

All shapes OK


In [12]:
import xarray as xr
from meridian.data.input_data import InputData

geo_coords = ['national']
time_coords = df['week'].dt.strftime('%Y-%m-%d').tolist()
channel_names = ['Sponsored_Products', 'Sponsored_Brands', 'DSP']

kpi_da = xr.DataArray(
    kpi,
    dims=['geo', 'time'],
    coords={'geo': geo_coords, 'time': time_coords},
    name='kpi'
)

media_da = xr.DataArray(
    media,
    dims=['geo', 'media_time', 'media_channel'],
    coords={'geo': geo_coords, 'media_time': time_coords, 'media_channel': channel_names},
    name='media'
)

media_spend_da = xr.DataArray(
    media_spend,
    dims=['geo', 'time', 'media_channel'],
    coords={'geo': geo_coords, 'time': time_coords, 'media_channel': channel_names},
    name='media_spend'
)

population_da = xr.DataArray(
    np.array([1.0]),
    dims=['geo'],
    coords={'geo': geo_coords},
    name='population'
)

input_data = InputData(
    kpi=kpi_da,
    kpi_type='revenue',
    population=population_da,
    media=media_da,
    media_spend=media_spend_da,
)

print("InputData created successfully")
print(f"N geos: {len(input_data.geo)}")
print(f"N time periods: {len(input_data.time)}")
print(f"N media channels: {len(input_data.media_channel)}")

InputData created successfully
N geos: 1
N time periods: 104
N media channels: 3


/usr/local/lib/python3.12/dist-packages/meridian/data/input_data.py:517: UserWarning: Revenue from the `kpi` data is used when `kpi_type`=`revenue`. `revenue_per_kpi` is ignored.
  warnings.warn(


In [13]:
from meridian.model import model, spec
from meridian.model import prior_distribution
import tensorflow_probability as tfp

roi_mu = 0.2
roi_sigma = 0.9

prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name='roi_m')
)

model_spec = spec.ModelSpec(prior=prior)

mmm = model.Meridian(input_data=input_data, model_spec=model_spec)

print("Model initialized successfully")

mmm.sample_prior(500, seed=0)
print("Prior sampling complete")

/usr/local/lib/python3.12/dist-packages/meridian/model/model.py:103: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1329: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. tau_g_excl_baseline has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1329: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_m has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1329: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_rf has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-p

Model initialized successfully
Prior sampling complete


In [14]:
print("Starting posterior sampling... (this takes ~10-15 min with GPU)")

mmm.sample_posterior(
    n_chains=4,
    n_adapt=500,
    n_burnin=500,
    n_keep=1000,
    seed=0
)

print("Posterior sampling complete!")
print(f"Inference data: {mmm.inference_data}")

Starting posterior sampling... (this takes ~10-15 min with GPU)
Posterior sampling complete!
Inference data: Inference data with groups:
	> posterior
	> sample_stats
	> prior
	> trace


/usr/local/lib/python3.12/dist-packages/arviz/data/inference_data.py:157: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/arviz/data/inference_data.py:1647: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(


In [15]:
from meridian.analysis import analyzer

# Crear analizador
mmm_analyzer = analyzer.Analyzer(mmm)

# R-hat - diagnostico de convergencia
print("=== CONVERGENCE DIAGNOSTICS ===")
rhat = mmm.inference_data.sample_stats['lp'].values
print(f"Log posterior R-hat check complete")

# ROI por canal
print("\n=== ROI BY CHANNEL (posterior median) ===")
summary = mmm_analyzer.summary_metrics()
print(summary)

/tmp/ipykernel_562/1823755419.py:4: DeprecationWarning: The `meridian` argument is deprecated and will be removed in a future version. Use `model_context` instead.
  mmm_analyzer = analyzer.Analyzer(mmm)


=== CONVERGENCE DIAGNOSTICS ===
Log posterior R-hat check complete

=== ROI BY CHANNEL (posterior median) ===
<xarray.Dataset> Size: 1kB
Dimensions:              (channel: 4, metric: 4, distribution: 2)
Coordinates:
  * channel              (channel) <U18 288B 'Sponsored_Products' ... 'All Ch...
  * metric               (metric) <U6 96B 'mean' 'median' 'ci_lo' 'ci_hi'
  * distribution         (distribution) <U9 72B 'prior' 'posterior'
Data variables:
    impressions          (channel) float32 16B 3.041e+05 1.043e+05 ... 5.327e+05
    pct_of_impressions   (channel) float32 16B 57.09 19.59 23.32 100.0
    spend                (channel) float32 16B 3.041e+05 1.043e+05 ... 5.327e+05
    pct_of_spend         (channel) float32 16B 57.09 19.59 23.32 100.0
    cpm                  (channel) float32 16B 1e+03 1e+03 1e+03 1e+03
    incremental_outcome  (channel, metric, distribution) float32 128B 5.726e+...
    pct_of_contribution  (channel, metric, distribution) float32 128B 7.536 ....
    roi 

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:352: UserWarning: Setting `use_kpi=True` has no effect when `kpi_type=REVENUE` since in this case, KPI is equal to revenue.
  warnings.warn(


In [16]:
import warnings
warnings.filterwarnings('ignore')

# Extraer ROI posterior por canal
roi_data = summary['roi']

print("=== ROI BY CHANNEL (Bayesian Posterior) ===\n")
channels = ['Sponsored_Products', 'Sponsored_Brands', 'DSP']

for ch in channels:
    mean  = float(roi_data.sel(channel=ch, metric='mean',   distribution='posterior'))
    median= float(roi_data.sel(channel=ch, metric='median', distribution='posterior'))
    ci_lo = float(roi_data.sel(channel=ch, metric='ci_lo',  distribution='posterior'))
    ci_hi = float(roi_data.sel(channel=ch, metric='ci_hi',  distribution='posterior'))
    print(f"{ch}:")
    print(f"  Median ROI:  ${median:.2f} per $1 spent")
    print(f"  Mean ROI:    ${mean:.2f} per $1 spent")
    print(f"  90% CI:      ${ci_lo:.2f} -- ${ci_hi:.2f}")
    print()

# Contribucion por canal
print("=== CONTRIBUTION BY CHANNEL (%) ===\n")
contrib = summary['pct_of_contribution']
for ch in channels:
    median = float(contrib.sel(channel=ch, metric='median', distribution='posterior'))
    ci_lo  = float(contrib.sel(channel=ch, metric='ci_lo',  distribution='posterior'))
    ci_hi  = float(contrib.sel(channel=ch, metric='ci_hi',  distribution='posterior'))
    print(f"{ch}: {median:.1f}% (90% CI: {ci_lo:.1f}% -- {ci_hi:.1f}%)")

=== ROI BY CHANNEL (Bayesian Posterior) ===

Sponsored_Products:
  Median ROI:  $2.93 per $1 spent
  Mean ROI:    $3.12 per $1 spent
  90% CI:      $0.61 -- $6.30

Sponsored_Brands:
  Median ROI:  $2.11 per $1 spent
  Mean ROI:    $3.27 per $1 spent
  90% CI:      $0.38 -- $10.26

DSP:
  Median ROI:  $4.08 per $1 spent
  Mean ROI:    $5.89 per $1 spent
  90% CI:      $0.44 -- $15.87

=== CONTRIBUTION BY CHANNEL (%) ===

Sponsored_Products: 13.4% (90% CI: 2.8% -- 28.9%)
Sponsored_Brands: 3.3% (90% CI: 0.6% -- 16.1%)
DSP: 7.6% (90% CI: 0.8% -- 29.7%)


In [18]:
from meridian.analysis import visualizer

# Response curves
media_effects = visualizer.MediaEffects(mmm)
fig = media_effects.plot_response_curves(
    plot_separately=True,
    include_ci=True
)
fig.show()

alt.FacetChart(...)

In [20]:
from meridian.analysis import optimizer

budget_opt = optimizer.BudgetOptimizer(mmm)

# Optimizar con presupuesto historico por defecto
results = budget_opt.optimize()

print("Optimization complete")

# Grafico de allocacion
fig = results.plot_budget_allocation()
fig.show()

Optimization complete


alt.Chart(...)

In [21]:
fig2 = results.plot_spend_delta()
fig2.show()

alt.LayerChart(...)

In [22]:
from google.colab import drive
drive.mount('/content/drive')

mmm.inference_data.to_netcdf('/content/drive/MyDrive/meridian_inference_data.nc')
print("Model saved to Drive")

Mounted at /content/drive
Model saved to Drive
